In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%pip install -q ultralytics==8.4.218

In [ ]:
import os
import cv2
import json
import torch
import hashlib
import shutil
import time
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
from ultralytics import YOLO

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

assert PROJECT_ROOT.is_dir()

SRC_DIR = PROJECT_ROOT / "src"

APP_DIR = PROJECT_ROOT / "app"

CONTEXT_DIR = PROJECT_ROOT / "models/context"

CONFIG_DIR = PROJECT_ROOT / "configs"

PIPELINE_RESULTS = (
    PROJECT_ROOT / "results/pipeline_testing"
)

EXTERNAL_DIR = (
    PROJECT_ROOT / "datasets/external_testing"
)

for folder in [
    SRC_DIR,
    APP_DIR,
    CONTEXT_DIR,
    CONFIG_DIR,
    PIPELINE_RESULTS,
    EXTERNAL_DIR / "helmet",
    EXTERNAL_DIR / "seatbelt",
    EXTERNAL_DIR / "mixed"
]:
    folder.mkdir(parents=True, exist_ok=True)


def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024**2), b""
        ):
            sha.update(chunk)

    return sha.hexdigest()


CONFIG_SOURCE = (
    PROJECT_ROOT /
    "results/final_evaluation/reports/"
    "final_inference_config.json"
)

assert CONFIG_SOURCE.is_file()

INFERENCE_CONFIG = json.loads(
    CONFIG_SOURCE.read_text()
)

MODEL_PATHS = {
    "helmet": PROJECT_ROOT /
              "models/helmet/helmet_best.pt",

    "seatbelt": PROJECT_ROOT /
                "models/seatbelt/seatbelt_best.pt"
}

for task, path in MODEL_PATHS.items():

    assert path.is_file(), path

    expected_hash = (
        INFERENCE_CONFIG["models"][task]
        ["model_sha256"]
    )

    assert sha256_file(path) == expected_hash

    print(f"{task.upper()} MODEL VERIFIED")

    print(
        "Confidence:",
        INFERENCE_CONFIG["models"][task]
        ["confidence"]
    )


DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("\nDevice:", DEVICE)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

print("\nPHASE 9 ENVIRONMENT READY")

In [ ]:
CONTEXT_WEIGHTS = (
    CONTEXT_DIR / "vehicle_context_yolov8n.pt"
)

if not CONTEXT_WEIGHTS.is_file():

    pretrained = YOLO("yolov8n.pt")

    source = Path(pretrained.ckpt_path)

    assert source.is_file()

    shutil.copy2(source, CONTEXT_WEIGHTS)

    del pretrained


context_model = YOLO(str(CONTEXT_WEIGHTS))

expected_classes = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

for cls, name in expected_classes.items():

    assert context_model.names[cls] == name

    print(cls, ":", name)

context_metadata = {
    "architecture": "YOLOv8n",
    "purpose": "Vehicle context association",
    "pretrained_dataset": "COCO",
    "custom_training": False,
    "weights": CONTEXT_WEIGHTS.name,
    "sha256": sha256_file(CONTEXT_WEIGHTS)
}

metadata_path = CONTEXT_WEIGHTS.with_suffix(".json")

if metadata_path.exists():
    assert json.loads(
        metadata_path.read_text()
    ) == context_metadata
else:
    metadata_path.write_text(
        json.dumps(context_metadata, indent=4)
    )

print("\nCONTEXT MODEL READY")

print("Weights:", CONTEXT_WEIGHTS)
print("SHA256:", context_metadata["sha256"])

del context_model

In [ ]:
CONTEXT_WEIGHTS = (
    CONTEXT_DIR / "vehicle_context_yolov8n.pt"
)

if not CONTEXT_WEIGHTS.is_file():

    pretrained = YOLO("yolov8n.pt")

    source = Path(pretrained.ckpt_path)

    assert source.is_file()

    shutil.copy2(source, CONTEXT_WEIGHTS)

    del pretrained


context_model = YOLO(str(CONTEXT_WEIGHTS))

expected_classes = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

for cls, name in expected_classes.items():

    assert context_model.names[cls] == name

    print(cls, ":", name)

context_metadata = {
    "architecture": "YOLOv8n",
    "purpose": "Vehicle context association",
    "pretrained_dataset": "COCO",
    "custom_training": False,
    "weights": CONTEXT_WEIGHTS.name,
    "sha256": sha256_file(CONTEXT_WEIGHTS)
}

metadata_path = CONTEXT_WEIGHTS.with_suffix(".json")

if metadata_path.exists():
    assert json.loads(
        metadata_path.read_text()
    ) == context_metadata
else:
    metadata_path.write_text(
        json.dumps(context_metadata, indent=4)
    )

print("\nCONTEXT MODEL READY")

print("Weights:", CONTEXT_WEIGHTS)
print("SHA256:", context_metadata["sha256"])

del context_model

In [ ]:
CONTEXT_WEIGHTS = (
    CONTEXT_DIR / "vehicle_context_yolov8n.pt"
)

if not CONTEXT_WEIGHTS.is_file():

    pretrained = YOLO("yolov8n.pt")

    source = Path(pretrained.ckpt_path)

    assert source.is_file()

    shutil.copy2(source, CONTEXT_WEIGHTS)

    del pretrained


context_model = YOLO(str(CONTEXT_WEIGHTS))

expected_classes = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

for cls, name in expected_classes.items():

    assert context_model.names[cls] == name

    print(cls, ":", name)

context_metadata = {
    "architecture": "YOLOv8n",
    "purpose": "Vehicle context association",
    "pretrained_dataset": "COCO",
    "custom_training": False,
    "weights": CONTEXT_WEIGHTS.name,
    "sha256": sha256_file(CONTEXT_WEIGHTS)
}

metadata_path = CONTEXT_WEIGHTS.with_suffix(".json")

if metadata_path.exists():
    assert json.loads(
        metadata_path.read_text()
    ) == context_metadata
else:
    metadata_path.write_text(
        json.dumps(context_metadata, indent=4)
    )

print("\nCONTEXT MODEL READY")

print("Weights:", CONTEXT_WEIGHTS)
print("SHA256:", context_metadata["sha256"])

del context_model

In [ ]:
APP_CONFIG = {
    "project": "TrafficGuard AI",

    "models": {

        "helmet": {
            "weights":
                "models/helmet/helmet_best.pt",

            **INFERENCE_CONFIG["models"]["helmet"]
        },

        "seatbelt": {
            "weights":
                "models/seatbelt/seatbelt_best.pt",

            **INFERENCE_CONFIG["models"]["seatbelt"]
        },

        "context": {
            "weights":
                "models/context/vehicle_context_yolov8n.pt",

            "model_sha256":
                sha256_file(CONTEXT_WEIGHTS),

            "image_size": 640,

            "confidence": 0.25,

            "nms_iou": 0.7,

            "classes": [0, 1, 2, 3, 5, 7]
        }
    },

    "context_association": {
        "enabled": True,

        "person_head_coverage": 0.5,

        "person_vehicle_horizontal_overlap": 0.3,

        "vehicle_occupant_coverage": 0.5
    },

    "video": {
        "frame_stride": 3,

        "counting_method":
            "per_processed_frame"
    },

    "application": {
        "automatic_enforcement": False,

        "human_review_required": True
    }
}

APP_CONFIG_PATH = (
    CONFIG_DIR / "app_config.json"
)

if APP_CONFIG_PATH.exists():

    assert json.loads(
        APP_CONFIG_PATH.read_text()
    ) == APP_CONFIG

else:

    APP_CONFIG_PATH.write_text(
        json.dumps(APP_CONFIG, indent=4)
    )

print(APP_CONFIG_PATH.read_text())

print("\nAPPLICATION CONFIGURATION SAVED")

In [ ]:
ENGINE_CODE = r'''
import cv2
import json
import time
import hashlib
import threading
import numpy as np
import torch

from pathlib import Path
from ultralytics import YOLO
from collections import Counter


class TrafficGuardEngine:

    def __init__(self, project_root, device=None):

        self.root = Path(project_root).resolve()

        config_path = (
            self.root / "configs/app_config.json"
        )

        self.config = json.loads(
            config_path.read_text()
        )

        self.device = (
            device if device is not None
            else (0 if torch.cuda.is_available() else "cpu")
        )

        self.models = {}
        self.lock = threading.RLock()

        expected_names = {
            "helmet": {
                0: "with_helmet",
                1: "without_helmet"
            },
            "seatbelt": {
                0: "with_seatbelt",
                1: "without_seatbelt"
            }
        }

        for task, cfg in self.config["models"].items():

            path = self.root / cfg["weights"]

            if not path.is_file():
                raise FileNotFoundError(path)

            sha = hashlib.sha256()

            with open(path, "rb") as f:
                for chunk in iter(
                    lambda: f.read(1024**2), b""
                ):
                    sha.update(chunk)

            if sha.hexdigest() != cfg["model_sha256"]:
                raise ValueError(
                    f"Model integrity failed: {task}"
                )

            self.models[task] = YOLO(str(path))

            if task in expected_names:

                if self.models[task].names != expected_names[task]:
                    raise ValueError(
                        f"Class mapping mismatch: {task}"
                    )


    @staticmethod
    def area(box):

        return (
            max(0, box[2] - box[0])
            * max(0, box[3] - box[1])
        )


    @staticmethod
    def intersection(a, b):

        return (
            max(
                0,
                min(a[2], b[2]) - max(a[0], b[0])
            )
            *
            max(
                0,
                min(a[3], b[3]) - max(a[1], b[1])
            )
        )


    @classmethod
    def coverage(cls, inner, outer):

        return cls.intersection(inner, outer) / max(
            cls.area(inner), 1e-9
        )


    @staticmethod
    def horizontal_overlap(a, b):

        overlap = max(
            0,
            min(a[2], b[2]) - max(a[0], b[0])
        )

        return overlap / max(
            min(a[2]-a[0], b[2]-b[0]),
            1e-9
        )


    def predict_model(self, task, image):

        cfg = self.config["models"][task]

        kwargs = {
            "source": image,
            "imgsz": cfg["image_size"],
            "conf": cfg["confidence"],
            "iou": cfg["nms_iou"],
            "max_det": cfg.get("max_det", 300),
            "device": self.device,
            "batch": 1,
            "verbose": False,
            "save": False,
            "stream": False
        }

        if task == "context":
            kwargs["classes"] = cfg["classes"]

        with self.lock, torch.inference_mode():

            result = self.models[task].predict(
                **kwargs
            )[0]

            detections = []

            if result.boxes is not None:

                for row in (
                    result.boxes.data
                    .detach().cpu().numpy()
                ):

                    x1, y1, x2, y2, conf, cls = row

                    cls = int(cls)

                    detections.append({
                        "task": task,
                        "class_id": cls,
                        "label": self.models[task].names[cls],
                        "confidence": float(conf),
                        "bbox": [
                            float(x1), float(y1),
                            float(x2), float(y2)
                        ]
                    })

        return detections


    def associate_helmet(self, detection, context):

        head = detection["bbox"]

        people = [
            d for d in context
            if d["class_id"] == 0
        ]

        motorcycles = [
            d for d in context
            if d["class_id"] == 3
        ]

        settings = self.config["context_association"]

        matches = []

        for person in people:

            body = person["bbox"]

            body_height = body[3] - body[1]

            head_y = (head[1] + head[3]) / 2

            # The helmet/head must be located
            # in the upper portion of a person.

            if head_y > body[1] + 0.45 * body_height:
                continue

            if self.coverage(head, body) < (
                settings["person_head_coverage"]
            ):
                continue

            for motorcycle in motorcycles:

                bike = motorcycle["bbox"]

                if self.horizontal_overlap(body, bike) < (
                    settings[
                        "person_vehicle_horizontal_overlap"
                    ]
                ):
                    continue

                # Require the person's lower body
                # to overlap the motorcycle region.

                lower_body = [
                    body[0],
                    body[1] + 0.45 * body_height,
                    body[2],
                    body[3]
                ]

                if self.intersection(
                    lower_body, bike
                ) <= 0:
                    continue

                matches.append(motorcycle)

        if matches:
            return "motorcycle_context_supported"

        return "context_unverified"


    def associate_seatbelt(self, detection, context):

        occupant = detection["bbox"]

        vehicles = [
            d for d in context
            if d["class_id"] in [2, 5, 7]
        ]

        minimum = self.config[
            "context_association"
        ]["vehicle_occupant_coverage"]

        for vehicle in vehicles:

            if self.coverage(
                occupant,
                vehicle["bbox"]
            ) >= minimum:

                return "vehicle_context_supported"

        return "context_unverified"


    def annotate(self, image, detections):

        output = image.copy()

        height, width = output.shape[:2]

        for d in detections:

            x1, y1, x2, y2 = map(
                int, d["bbox"]
            )

            x1 = max(0, min(width-1, x1))
            x2 = max(0, min(width-1, x2))
            y1 = max(0, min(height-1, y1))
            y2 = max(0, min(height-1, y2))

            if d["status"] == "context_unverified":

                color = (0, 165, 255)

            elif d["class_id"] == 1:

                color = (0, 0, 230)

            else:

                color = (0, 180, 0)

            label = (
                f'{d["label"]} '
                f'{d["confidence"]:.2f}'
            )

            if d["status"] == "context_unverified":
                label += " | review"

            elif d["class_id"] == 1:
                label += " | possible"

            cv2.rectangle(
                output,
                (x1, y1),
                (x2, y2),
                color,
                2
            )

            cv2.putText(
                output,
                label,
                (x1, max(15, y1-5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                color,
                1,
                cv2.LINE_AA
            )

        return output


    def process_image(
        self,
        image,
        mode="combined",
        use_context=True
    ):

        if (
            image is None
            or not isinstance(image, np.ndarray)
            or image.size == 0
            or image.ndim != 3
            or image.shape[2] != 3
        ):
            raise ValueError(
                "Expected a valid three-channel BGR image."
            )

        if mode not in {
            "helmet", "seatbelt", "combined"
        }:
            raise ValueError("Unsupported detection mode")

        start = time.perf_counter()

        context = (
            self.predict_model("context", image)
            if use_context else []
        )

        tasks = (
            ["helmet", "seatbelt"]
            if mode == "combined"
            else [mode]
        )

        detections = []

        for task in tasks:

            rows = self.predict_model(task, image)

            for row in rows:

                if not use_context:

                    row["status"] = "context_unverified"

                elif task == "helmet":

                    row["status"] = (
                        self.associate_helmet(
                            row, context
                        )
                    )

                else:

                    row["status"] = (
                        self.associate_seatbelt(
                            row, context
                        )
                    )

                row["possible_violation"] = (
                    row["class_id"] == 1
                    and row["status"]
                    != "context_unverified"
                )

                detections.append(row)

        counts = Counter(
            d["label"] for d in detections
        )

        summary = {
            "mode": mode,
            "total_detections": len(detections),

            "with_helmet":
                counts["with_helmet"],

            "without_helmet":
                counts["without_helmet"],

            "with_seatbelt":
                counts["with_seatbelt"],

            "without_seatbelt":
                counts["without_seatbelt"],

            "possible_helmet_violations":
                sum(
                    d["possible_violation"]
                    and d["task"] == "helmet"
                    for d in detections
                ),

            "possible_seatbelt_violations":
                sum(
                    d["possible_violation"]
                    and d["task"] == "seatbelt"
                    for d in detections
                ),

            "unverified_detections":
                sum(
                    d["status"] == "context_unverified"
                    for d in detections
                )
        }

        annotated = self.annotate(
            image, detections
        )

        summary["processing_ms"] = (
            time.perf_counter() - start
        ) * 1000

        return {
            "annotated_image": annotated,
            "detections": detections,
            "context_objects": context,
            "summary": summary
        }
'''

ENGINE_PATH = SRC_DIR / "trafficguard_engine.py"

# Verify Python syntax before writing the file.
compile(ENGINE_CODE, str(ENGINE_PATH), "exec")

if ENGINE_PATH.exists():

    assert ENGINE_PATH.read_text() == ENGINE_CODE, (
        "A different engine already exists. "
        "Preserve it before making changes."
    )

else:

    ENGINE_PATH.write_text(
        ENGINE_CODE,
        encoding="utf-8"
    )

print("Python syntax: PASSED")

print("Engine created:", ENGINE_PATH)

In [ ]:
import importlib

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import trafficguard_engine

importlib.reload(trafficguard_engine)

from trafficguard_engine import TrafficGuardEngine

engine = TrafficGuardEngine(
    project_root=PROJECT_ROOT,
    device=DEVICE
)

print("=" * 60)
print("TRAFFICGUARD INFERENCE ENGINE")
print("=" * 60)

for task, model in engine.models.items():

    print("\nModel:", task)

    if task != "context":
        print("Classes:", model.names)

    print(
        "Confidence:",
        engine.config["models"][task]
        ["confidence"]
    )

print("\nAll three models loaded successfully.")
print("Device:", engine.device)

In [ ]:
from google.colab import files

uploaded = files.upload()

assert len(uploaded) > 0

SESSION_ID = datetime.now().strftime(
    "%Y%m%d-%H%M%S-%f"
)

IMAGE_TEST_DIR = (
    PIPELINE_RESULTS / f"images_{SESSION_ID}"
)

IMAGE_TEST_DIR.mkdir(
    parents=True,
    exist_ok=False
)

all_summaries = []
all_detections = []

for index, (filename, content) in enumerate(
    uploaded.items()
):

    array = np.frombuffer(
        content,
        dtype=np.uint8
    )

    image = cv2.imdecode(
        array,
        cv2.IMREAD_COLOR
    )

    if image is None:
        print("Skipped invalid image:", filename)
        continue

    result = engine.process_image(
        image,
        mode="combined",
        use_context=True
    )

    summary = {
        "Image": filename,
        **result["summary"]
    }

    all_summaries.append(summary)

    for detection in result["detections"]:

        all_detections.append({
            "Image": filename,
            **detection
        })

    output_path = (
        IMAGE_TEST_DIR /
        f"{index:03d}_{Path(filename).stem}_detected.jpg"
    )

    assert cv2.imwrite(
        str(output_path),
        result["annotated_image"]
    )

    fig, axes = plt.subplots(
        1, 2,
        figsize=(15, 7)
    )

    axes[0].imshow(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )

    axes[1].imshow(
        cv2.cvtColor(
            result["annotated_image"],
            cv2.COLOR_BGR2RGB
        )
    )

    axes[0].set_title("Original Image")
    axes[1].set_title("TrafficGuard Detection")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    print(json.dumps(summary, indent=4))


summary_df = pd.DataFrame(all_summaries)

summary_df.to_csv(
    IMAGE_TEST_DIR / "image_summary.csv",
    index=False
)

(
    IMAGE_TEST_DIR / "detections.json"
).write_text(
    json.dumps(all_detections, indent=4)
)

display(summary_df)

print("\nImage results:", IMAGE_TEST_DIR)

In [ ]:
test_cases = {
    "Black Image": np.zeros(
        (640, 640, 3),
        dtype=np.uint8
    ),

    "White Image": np.full(
        (640, 640, 3),
        255,
        dtype=np.uint8
    )
}

test_records = []

for name, image in test_cases.items():

    result = engine.process_image(
        image,
        mode="combined"
    )

    total = result["summary"][
        "total_detections"
    ]

    test_records.append({
        "Test": name,
        "Expected Detections": 0,
        "Actual Detections": total,
        "Status": (
            "PASS" if total == 0 else "REVIEW"
        )
    })


try:

    engine.process_image(None)

    status = "FAIL"

except ValueError:

    status = "PASS"

test_records.append({
    "Test": "Invalid Image Input",
    "Status": status
})


try:

    engine.process_image(
        test_cases["Black Image"],
        mode="invalid"
    )

    status = "FAIL"

except ValueError:

    status = "PASS"

test_records.append({
    "Test": "Invalid Detection Mode",
    "Status": status
})

system_test_df = pd.DataFrame(test_records)

display(system_test_df)

system_test_df.to_csv(
    IMAGE_TEST_DIR / "basic_system_tests.csv",
    index=False
)

In [ ]:
VIDEO_CODE = r'''
import cv2
import json
import math
import time
import csv

from pathlib import Path


def process_video(
    engine,
    input_path,
    output_path,
    mode="combined",
    frame_stride=3,
    max_frames=None,
    progress_callback=None
):

    input_path = Path(input_path)
    output_path = Path(output_path)

    if not input_path.is_file():
        raise FileNotFoundError(input_path)

    if output_path.exists():
        raise FileExistsError(output_path)

    if frame_stride < 1:
        raise ValueError("Invalid frame stride")

    cap = cv2.VideoCapture(str(input_path))

    if not cap.isOpened():
        cap.release()
        raise ValueError("Unable to open video")

    source_fps = cap.get(cv2.CAP_PROP_FPS)

    if not math.isfinite(source_fps) or source_fps <= 0:
        cap.release()
        raise ValueError("Invalid source video FPS")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if width <= 0 or height <= 0:
        cap.release()
        raise ValueError("Invalid video dimensions")

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    output_fps = source_fps / frame_stride

    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        output_fps,
        (width, height)
    )

    if not writer.isOpened():
        cap.release()
        writer.release()
        raise RuntimeError("Unable to create video output")

    frame_index = 0
    processed = 0
    records = []

    start = time.perf_counter()

    try:

        while True:

            ok, frame = cap.read()

            if not ok:
                break

            if frame_index % frame_stride == 0:

                result = engine.process_image(
                    frame,
                    mode=mode,
                    use_context=True
                )

                writer.write(
                    result["annotated_image"]
                )

                records.append({
                    "frame": frame_index,
                    "timestamp_seconds":
                        frame_index / source_fps,
                    **result["summary"]
                })

                processed += 1

                if progress_callback is not None:

                    progress_callback(
                        frame_index + 1,
                        total_frames
                    )

                if (
                    max_frames is not None
                    and processed >= max_frames
                ):
                    break

            frame_index += 1

    finally:

        cap.release()
        writer.release()

    if not records:
        raise RuntimeError("No frames processed")

    elapsed = time.perf_counter() - start

    report = {
        "source_video": input_path.name,

        "source_fps": source_fps,

        "output_fps": output_fps,

        "frame_stride": frame_stride,

        "processed_frames": processed,

        "processing_seconds": elapsed,

        "processed_frames_per_second":
            processed / max(elapsed, 1e-9),

        "helmet_violation_detections":
            sum(
                r["possible_helmet_violations"]
                for r in records
            ),

        "seatbelt_violation_detections":
            sum(
                r["possible_seatbelt_violations"]
                for r in records
            ),

        "unverified_detections":
            sum(
                r["unverified_detections"]
                for r in records
            ),

        "counting_method":
            "per_processed_frame",

        "unique_violation_count": None
    }

    csv_path = output_path.with_suffix(".csv")

    with open(
        csv_path, "w",
        newline="", encoding="utf-8"
    ) as f:

        writer_csv = csv.DictWriter(
            f,
            fieldnames=list(records[0].keys())
        )

        writer_csv.writeheader()
        writer_csv.writerows(records)

    output_path.with_suffix(".json").write_text(
        json.dumps(report, indent=4)
    )

    return report
'''

VIDEO_MODULE = SRC_DIR / "video_processor.py"

compile(VIDEO_CODE, str(VIDEO_MODULE), "exec")

if VIDEO_MODULE.exists():

    assert VIDEO_MODULE.read_text() == VIDEO_CODE

else:

    VIDEO_MODULE.write_text(
        VIDEO_CODE,
        encoding="utf-8"
    )

print("Video processor created:", VIDEO_MODULE)
print("Python syntax: PASSED")

In [ ]:
from google.colab import files
from tqdm.auto import tqdm

from video_processor import process_video

uploaded_videos = files.upload()

assert len(uploaded_videos) == 1, (
    "Upload one short video."
)

filename, content = next(
    iter(uploaded_videos.items())
)

suffix = Path(filename).suffix.lower()

assert suffix in {
    ".mp4", ".avi", ".mov", ".mkv", ".webm"
}

VIDEO_SESSION = datetime.now().strftime(
    "%Y%m%d-%H%M%S-%f"
)

VIDEO_TEST_DIR = (
    PIPELINE_RESULTS /
    f"video_{VIDEO_SESSION}"
)

VIDEO_TEST_DIR.mkdir(
    parents=True,
    exist_ok=False
)

INPUT_VIDEO = (
    VIDEO_TEST_DIR / f"input{suffix}"
)

INPUT_VIDEO.write_bytes(content)

OUTPUT_VIDEO = (
    VIDEO_TEST_DIR /
    "trafficguard_output.mp4"
)

report = process_video(

    engine=engine,

    input_path=INPUT_VIDEO,

    output_path=OUTPUT_VIDEO,

    mode="combined",

    frame_stride=3,

    max_frames=60

)

print(json.dumps(report, indent=4))

print("\nVIDEO PROCESSING COMPLETED")

print("Video:", OUTPUT_VIDEO)

print(
    "Frame statistics:",
    OUTPUT_VIDEO.with_suffix(".csv")
)

In [ ]:
import subprocess

from IPython.display import Video, display

PLAYABLE_VIDEO = (
    VIDEO_TEST_DIR /
    "trafficguard_output_h264.mp4"
)

if shutil.which("ffmpeg"):

    subprocess.run(
        [
            "ffmpeg",
            "-n",
            "-i", str(OUTPUT_VIDEO),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(PLAYABLE_VIDEO)
        ],
        check=True,
        capture_output=True
    )

    display(
        Video(
            str(PLAYABLE_VIDEO),
            embed=True,
            width=850
        )
    )

else:

    print(
        "FFmpeg unavailable. "
        "Open the saved video in Google Drive."
    )

In [ ]:
FRAME_CSV = OUTPUT_VIDEO.with_suffix(".csv")

assert FRAME_CSV.is_file()

video_df = pd.read_csv(FRAME_CSV)

display(video_df.head())

columns = [
    "possible_helmet_violations",
    "possible_seatbelt_violations",
    "unverified_detections"
]

available_columns = [
    c for c in columns
    if c in video_df.columns
]

fig, ax = plt.subplots(
    figsize=(14, 5)
)

for column in available_columns:

    ax.plot(
        video_df["timestamp_seconds"],
        video_df[column],
        marker=".",
        label=column.replace("_", " ").title()
    )

ax.set_title(
    "TrafficGuard AI — Detections Over Time"
)

ax.set_xlabel("Video time (seconds)")

ax.set_ylabel("Detections per processed frame")

ax.legend()

ax.grid(alpha=0.2)

plt.tight_layout()

plt.savefig(
    VIDEO_TEST_DIR /
    "video_detection_statistics.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

display(
    video_df[available_columns]
    .describe()
    .round(2)
)

In [ ]:

required_files = [

    PROJECT_ROOT /
    "src/trafficguard_engine.py",

    PROJECT_ROOT /
    "src/video_processor.py",

    PROJECT_ROOT /
    "configs/app_config.json",

    PROJECT_ROOT /
    "models/helmet/helmet_best.pt",

    PROJECT_ROOT /
    "models/seatbelt/seatbelt_best.pt",

    PROJECT_ROOT /
    "models/context/vehicle_context_yolov8n.pt"
]

verification = []

for path in required_files:

    verification.append({
        "File": str(
            path.relative_to(PROJECT_ROOT)
        ),
        "Exists": path.is_file(),
        "Size MB": round(
            path.stat().st_size / 1024**2,
            4
        ) if path.is_file() else 0
    })

verification_df = pd.DataFrame(
    verification
)

display(verification_df)

assert verification_df["Exists"].all()

# Ensure trained model weights remain unchanged.

for task, path in MODEL_PATHS.items():

    assert sha256_file(path) == (
        INFERENCE_CONFIG["models"][task]
        ["model_sha256"]
    )

assert json.loads(
    CONFIG_SOURCE.read_text()
) == INFERENCE_CONFIG


phase9_report = {
    "project": "TrafficGuard AI",

    "phase": "9",

    "created_at":
        datetime.now().isoformat(),

    "device": str(DEVICE),

    "training_performed": False,

    "helmet_model": "YOLOv8m",

    "seatbelt_model": "YOLOv8s",

    "context_model": "YOLOv8n",

    "helmet_confidence": 0.30,

    "seatbelt_confidence": 0.35,

    "image_pipeline_executed":
        bool(globals().get("all_summaries")),

    "video_pipeline_executed":
        bool(globals().get("report", {}).get(
            "processed_frames", 0
        )),

    "basic_system_tests":
        json.loads(
            system_test_df.to_json(
                orient="records"
            )
        ) if "system_test_df" in globals() else [],

    "original_weights_unchanged": True,

    "limitations": [
        "Context association is heuristic.",
        "External quantitative evaluation is pending.",
        "Video statistics count detection instances.",
        "Unique violation tracking is not implemented.",
        "All possible violations require human review."
    ]
}

REPORT_PATH = (
    PIPELINE_RESULTS /
    f"phase9_report_{datetime.now():%Y%m%d-%H%M%S-%f}.json"
)

REPORT_PATH.write_text(
    json.dumps(phase9_report, indent=4)
)

print(json.dumps(phase9_report, indent=4))

print("\n" + "=" * 60)
print("PHASE 9 FILE VERIFICATION PASSED")
print("=" * 60)

print("Helmet weights unchanged: YES")
print("Seatbelt weights unchanged: YES")
print("Original evaluation settings preserved: YES")
print("Inference engine: CREATED")
print("Video processor: CREATED")
print("Application configuration: CREATED")
print("Results stored in Google Drive: YES")

print("\nReview pipeline test outputs before Phase 10.")